<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/rl/02_environment_step.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# R2 環境を作る:reset() と step()(notebook)

本文:[`02-environment-step.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/rl/02-environment-step.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-8):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

# R2〜R5 で使うグリッド(本文 R2 §3)
N_ROW = N_COL = 4
S, A = N_ROW * N_COL, 4
START, GOAL, MUD = 0, 15, {5, 10}
MOVES = [(-1, 0), (0, 1), (1, 0), (0, -1)]   # 0=上、1=右、2=下、3=左
ARROWS = ["↑", "→", "↓", "←"]

## 1. 座標と番号

**手計算**:本文の確認課題1(マス 9 は何行何列か、`(3, 2)` は何番か)を解く。

- 手計算:(ここに書く)

**実装**:座標 `(行, 列)` をマスの番号にする `to_index` と、その逆の `to_pos` を書く(本文 §3)。

In [ ]:
def to_index(pos):
    r, c = pos
    raise NotImplementedError("TODO: 行 × 列の数 + 列")

def to_pos(s):
    raise NotImplementedError("TODO: (行, 列) を返す。divmod を使うとよい")

check("(1, 1) はマス 5", to_index((1, 1)), 5)
check("マス 11 は (2, 3)", to_pos(11), (2, 3))
check("16マスすべてで、行き来すると元に戻る", [to_index(to_pos(s)) for s in range(S)], list(range(S)))

## 2. 次のマス

**予想**:マス 3(右上の角)で「右」を選ぶと、どのマスに行くか。

- 予想:(ここに書く)

**実装**:マス `s` で行動 `a` をとったときの次のマスを返す関数 `next_state` を書く。外に出る方向なら、その場にとどまる。

In [ ]:
def next_state(s, a):
    r, c = to_pos(s)
    dr, dc = MOVES[a]
    raise NotImplementedError("TODO: (r + dr, c + dc) がグリッドの中なら、そのマスの番号。外なら s のまま")

check("マス0 で上:外なので 0", next_state(0, 0), 0)
check("マス0 で右:1", next_state(0, 1), 1)
check("マス3 で右:外なので 3", next_state(3, 1), 3)
check("マス5 で下:9", next_state(5, 2), 9)
check("マス12 で左:外なので 12", next_state(12, 3), 12)

## 3. 環境のクラスを作る

`reset()` は用意してある。**実装**:`step(action)` を、本文 §4 の順に書く。

- 確率 `self.slip` で、行動を `self.rng.integers(A)`(4方向からランダム)に置き換える
- ぬかるみのマスに入ったら報酬 `self.mud_reward`(−10)、それ以外は −1
- ゴールに着いたら `terminated = True`。`self.max_steps` ステップで着かなければ `truncated = True`
- `self.s` と `self.t` を更新する

下の `check` は、滑らない床で決めた行動の列を動かして、通るマス・報酬・終わりの判定を確かめる。最後に、滑る床で「右へ進む確率」を数えて確かめる(本文の確認課題4)。

In [ ]:
class SlipperyGrid:
    """R2 のグリッドの環境(Gymnasium と同じ reset・step の約束)"""
    def __init__(self, slip=0.2, max_steps=100, mud_reward=-10.0):
        self.slip, self.max_steps, self.mud_reward = slip, max_steps, mud_reward
        self.rng = np.random.default_rng()
        self.s, self.t = START, 0

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)       # 同じ seed なら、同じ結果になる
        self.s, self.t = START, 0
        return self.s, {}

    def step(self, action):
        raise NotImplementedError("TODO: 本文 §4 の順に、(1) 確率 slip で行動をランダムな方向に置き換える "
                                  "(2) 次のマス (3) 報酬 (4) terminated・truncated (5) self.s を更新して5つの値を返す")

# 滑らない床で、決めた行動の列で動かして確かめる(本文の確認課題2・3)
env = SlipperyGrid(slip=0.0)
env.reset()
states, total = [], 0.0
for a in [1, 1, 1, 2, 2, 2]:                        # 右・右・右・下・下・下
    s2, r, terminated, truncated, info = env.step(a)
    states.append(s2); total += r
check("通るマス", states, [1, 2, 3, 7, 11, 15])
check("報酬の合計", total, -6.0)
check("最後に terminated", terminated, True)

env.reset()
check("スタートで上:その場にとどまる", env.step(0)[:2], [0, -1.0])
env.reset(); env.step(2)
check("マス4 から右:ぬかるみ(マス5)で報酬 -10", env.step(1)[:2], [5, -10.0])

env = SlipperyGrid(slip=0.0, max_steps=3)
env.reset()
for a in [1, 3, 1]:
    out = env.step(a)
check("3ステップで時間切れ:terminated=False, truncated=True", out[2:4], [False, True])

# 滑る床:マス0 で右を選んだとき、実際にマス1 へ進む割合(本文の確認課題4)
env = SlipperyGrid(slip=0.2)
env.reset(seed=0)
hits = 0
for _ in range(20000):
    env.reset()
    hits += env.step(1)[0] == 1
check("右へ進む確率 ≈ 0.8 + 0.2/4 = 0.85", hits / 20000, 0.85, tol=0.01)

## 4. ランダムな方策で動かす

**予想**:滑る床(`slip = 0.2`)で、でたらめに行動を選ぶと、ゴールまで平均何ステップかかるか。100 ステップ以内にゴールに着く割合はどれくらいか。

- 予想:(ここに書く)

**実装**:1回のエピソードを動かし、報酬の合計・ステップ数・ゴールに着いたか(`terminated`)を返す関数 `run_episode` を書く(本文 §1 のループ)。方策は、「状態と乱数を受け取って、行動を返す関数」として渡す。方策の例を4つ用意してある。

In [ ]:
def run_episode(env, policy, seed):
    raise NotImplementedError("TODO: reset(seed) してから、終わる(terminated か truncated)まで、"
                              "policy(s, env.rng) で行動を選んで step をくり返す。報酬の合計・ステップ数・terminated を返す")

def random_policy(s, rng):
    return int(rng.integers(A))

def right_then_down(s, rng):
    r, c = to_pos(s)
    return 1 if c < N_COL - 1 else 2        # 右の端まで右、そのあと下

def down_then_right(s, rng):
    r, c = to_pos(s)
    return 2 if r < N_ROW - 1 else 1        # 下の端まで下、そのあと右

def zigzag(s, rng):
    r, c = to_pos(s)
    return 2 if r <= c else 1               # 下と右を交互に(ぬかるみの上を通る)

check("滑らない床、右→下の方策:(-6, 6, True)", run_episode(SlipperyGrid(slip=0.0), right_then_down, seed=0), [-6, 6, True])
check("滑らない床、ジグザグ:ぬかるみ2つで -24", run_episode(SlipperyGrid(slip=0.0), zigzag, seed=0)[0], -24)

In [ ]:
env = SlipperyGrid(slip=0.2, max_steps=100)
results = np.array([run_episode(env, random_policy, seed) for seed in range(500)])
total, steps, reached = results[:, 0], results[:, 1], results[:, 2]
print(f"ランダムな方策(500 エピソード):報酬の合計 平均 {total.mean():.1f}、ステップ数 平均 {steps.mean():.1f}、"
      f"ゴールに着いた割合 {reached.mean():.2f}")
plt.figure(figsize=(5, 3))
plt.hist(total, bins=30)
plt.xlabel("total reward of an episode"); plt.ylabel("count"); plt.title("random policy, slip = 0.2")
plt.show()

## 5. 決めた方策を比べる

**予想**:滑る床(`slip = 0.2`)で、「右→下」「下→右」「ジグザグ」の3つの方策のうち、報酬の合計の平均が一番大きいのはどれか。「右→下」と「下→右」に差はあるか。

- 予想:(ここに書く)

1回のエピソードだけでは比べられない(本文 §5)。1000 個の seed で、平均と標準偏差を求める。

In [ ]:
def compare(policies, slip=0.2, n=1000, mud_reward=-10.0):
    env = SlipperyGrid(slip=slip, mud_reward=mud_reward)
    for name, policy in policies.items():
        total = np.array([run_episode(env, policy, seed)[0] for seed in range(n)])
        print(f"{name:12s} 平均 {total.mean():7.2f}  標準偏差 {total.std():5.2f}  "
              f"平均の標準誤差 {total.std() / np.sqrt(n):.2f}")

policies = {"右→下": right_then_down, "下→右": down_then_right, "ジグザグ": zigzag, "ランダム": random_policy}
compare(policies)

env = SlipperyGrid(slip=0.2)
print("seed 0 だけで比べると:", {name: run_episode(env, p, 0)[0] for name, p in policies.items()})

「右→下」と「下→右」は、グリッドの対称性から、平均はほぼ同じになる。差は、平均の標準誤差(およそ 0.15)と同じくらいである。1つの seed だけで比べると、どちらかが大きく見えることがある。

**再現性**:同じ seed なら、同じ結果になることを確かめる。

In [ ]:
env = SlipperyGrid(slip=0.2)
check("同じ seed で2回動かすと、同じ結果", run_episode(env, random_policy, 7), run_episode(env, random_policy, 7))
print("seed を変えると:", [run_episode(env, random_policy, seed)[0] for seed in range(5)])

## 6. 条件を変える

1. `slip` を 0、0.2、0.5 にして、`compare` で3つの方策を比べる。滑りやすくなると、どの方策が一番悪くなるか
2. ぬかるみの報酬を −10 から −2 にする(`compare(policies, mud_reward=-2.0)`)。ジグザグの方策の順位はどう変わるか
3. `max_steps` を 20 にして、ランダムな方策が時間切れ(`truncated`)になる割合を数える

In [ ]:
# 1〜3 をここで試す

## 7. 研究室のライブラリ `cpai_lab` との関係

研究では、研究室の共通ライブラリ `cpai_lab`([lab-core](https://github.com/oit-cpai/lab-core))の `GridMazeEnv` を使う。この単元で作った環境と同じ約束(`reset()` と `step()`、5つの返り値)で作ってある。文字で書いた地図(`#` = 壁、`S` = スタート、`G` = ゴール)から、迷路を作れる。

この単元のグリッドと同じ 4×4 の広場を、まわりを壁で囲んで作る。

In [ ]:
# 研究室のライブラリ cpai_lab を入れる(Colab では、ランタイムをつなぐたびに必要)
%pip install -q git+https://github.com/oit-cpai/lab-core.git

In [ ]:
from cpai_lab import GridMazeEnv
from cpai_lab.envs.grid_maze import grid_from_text

text = """
######
#S...#
#....#
#....#
#...G#
######
"""
grid, start, goal, traps = grid_from_text(text)
lab_env = GridMazeEnv(grid=grid, start=start, goal=goal, slip_prob=0.0, step_reward=-1.0, goal_reward=-1.0)
obs, info = lab_env.reset(seed=0)
print("最初の状態:", obs, info)

# cpai_lab の行動の番号は 0=上、1=下、2=左、3=右(この単元とは違う)
total = 0.0
for a in [3, 3, 3, 1, 1, 1]:                 # 右・右・右・下・下・下
    obs, r, terminated, truncated, info = lab_env.step(a)
    total += r
check("cpai_lab でも、右3回・下3回でゴール(報酬の合計 -6)", [total, terminated], [-6.0, True])
print("最後の状態:", obs, info)

この単元の環境と `GridMazeEnv` の違いをまとめる。研究で使うときは、ここに気をつける。

| | この単元の `SlipperyGrid` | `cpai_lab` の `GridMazeEnv` |
|---|---|---|
| 行動の番号 | 0=上、1=右、2=下、3=左 | **0=上、1=下、2=左、3=右** |
| 状態の番号 | 16マスに 0〜15 | 壁も含めた地図のマスの番号(この例では 6×6 なので、スタートは 1 行 1 列で 7) |
| 報酬 | 1ステップ −1、ぬかるみ −10 | `step_reward`(既定 −0.01)、ゴールで `goal_reward`(既定 +1)。落とし穴(`T`)は入ると終わる |
| 地図 | 4×4 に固定 | 文字の地図、または `generate_maze` で自動で作る |
| `info` | `slipped` | `pos`(座標)、`slipped`、`hit_wall`、`at_goal` など |

## 8. 説明(自分の言葉で書く)

1. 相互作用のループで、エージェントと環境がそれぞれ何をするか
   - (ここに書く)
2. `terminated` と `truncated` を区別しないと、何が困るか
   - (ここに書く)
3. 2つの方策を、1回のエピソードで比べてはいけないのはなぜか。どう比べればよいか
   - (ここに書く)